In [4]:
def RLE(b, Ms=8, Mc=8):
    if not b:
        return b''
    
    symbol_size = Ms // 8
    control_size = Mc // 8
    max_count = (1 << (control_size * 8 - 1)) - 1  # резервируем бит флага
    
    result = bytearray()
    i = 0
    n = len(b)
    
    while i < n:
        # Считаем повторы текущего символа
        current = b[i:i+symbol_size]
        count = 1
        j = i + symbol_size
        while j + symbol_size <= n and b[j:j+symbol_size] == current and count < max_count:
            count += 1
            j += symbol_size
        
        if count >= 3:
            result.extend(count.to_bytes(control_size, 'big'))
            result.extend(current)
            i = j
        else:
            # Собираем литералы (неповторяющиеся символы)
            literal_start = i
            while i < n:
                sym = b[i:i+symbol_size]
                if len(sym) < symbol_size:
                    break
                # Проверяем, не начался ли повтор из 3+ символов
                next_sym = b[i+symbol_size:i+2*symbol_size] if i + 2*symbol_size <= n else None
                if next_sym and sym == next_sym:
                    # Проверяем длину потенциального повтора
                    rep_count = 1
                    k = i + symbol_size
                    while k + symbol_size <= n and b[k:k+symbol_size] == sym and rep_count < max_count:
                        rep_count += 1
                        k += symbol_size
                    if rep_count >= 3:
                        break  # начался выгодный повтор
                i += symbol_size
                if (i - literal_start) // symbol_size >= max_count:
                    break
            
            # Записываем литералы
            literal_bytes = b[literal_start:i]
            lit_count = len(literal_bytes) // symbol_size
            if lit_count > 0:
                result.extend(((lit_count | 0x80) if control_size == 1 else lit_count).to_bytes(control_size, 'big'))
                result.extend(literal_bytes)
    
    return bytes(result)

def deRLE(b, Ms=8, Mc=8):
    if not b:
        return b''
    
    symbol_size = Ms // 8
    control_size = Mc // 8
    result = bytearray()
    i = 0
    
    while i < len(b):
        control = int.from_bytes(b[i:i+control_size], 'big')
        i += control_size
        
        if control & 0x80:  # литерал
            count = control & 0x7F
            end = i + count * symbol_size
            result.extend(b[i:end])
            i = end
        else:  # повтор
            symbol = b[i:i+symbol_size]
            i += symbol_size
            for _ in range(control):
                result.extend(symbol)
    
    return bytes(result)

def read_rle_file(filename):
    with open(filename, 'rb') as f:
        meta = f.read(2)
        Ms, Mc = meta[0], meta[1]
        data = f.read()
    return Ms, Mc, data

def write_rle_file(filename, data, Ms, Mc):
    with open(filename, 'wb') as f:
        f.write(bytes([Ms, Mc]))
        f.write(data)

def compress(input_path, output_path, Ms=16, Mc=16):
    with open(input_path, 'rb') as f:
        data = f.read()
    
    compressed = RLE(data, Ms, Mc)
    write_rle_file(output_path, compressed, Ms, Mc)

def decompress(input_path, output_path):
    Ms, Mc, compressed = read_rle_file(input_path)
    
    decompressed = deRLE(compressed, Ms, Mc)
    
    with open(output_path, 'wb') as f:
        f.write(decompressed)

TEST_FILES = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/enwik7",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test2.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test3.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test4.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test5.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/test6.raw",
]

TEST_FILES_COMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/enwik7_comp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test2_comp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test3_comp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test4_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test5_comp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test6_comp.raw",
]

TEST_FILES_DECOMPRESS = [
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/enwik7_decomp",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test2_decomp.txt",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test3_decomp.exe",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test4_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test5_decomp.raw",
    "C:/Users/irbis/Desktop/аисд 2 курс 2 сем/лаба 1/test_files/rle/test6_decomp.raw",
]

from pathlib import Path

for file_path in range(len(TEST_FILES)):
    
    original_size = Path(TEST_FILES[file_path]).stat().st_size
    
    print(f"\nФайл: {file_path+1} ({original_size:,} байт)")

    # Сжатие
    compress(TEST_FILES[file_path], TEST_FILES_COMPRESS[file_path])
    compressed_size = Path(TEST_FILES_COMPRESS[file_path]).stat().st_size
    
    # Распаковка
    decompress(TEST_FILES_COMPRESS[file_path], TEST_FILES_DECOMPRESS[file_path])
    decompressed_size = Path(TEST_FILES_DECOMPRESS[file_path]).stat().st_size
    
    # Проверка целостности
    if decompressed_size == original_size:
        print("correct")
    else:
        print("fail")
    
    # Статистика
    ratio = (original_size / compressed_size) if compressed_size > 0 else 0
    print(f"Исходный размер: {original_size}\n")
    print(f"После сжатия: {compressed_size}\n")
    print(f"После распаковки: {decompressed_size}\n")
    print(f"Коэффициент сжатия: {ratio}\n")


Файл: 1 (10,000,000 байт)


KeyboardInterrupt: 